#### Import Libraries

In [1]:
import argparse
import pandas as pd
import io
import argparse
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

#### Regional Summary

In [2]:
# regional_summary.py
# Aggregate municipal EV charger estimates (rf/xgb_EV_area_distr.csv) into the
# five Danish regions (NUTS 2, cf. https://www.dst.dk/en/Statistik/dokumentation/nomenklaturer/nuts)
# with a selectable probability threshold.
#
# Usage:
#   python regional_summary.py                      # both models, tau = 0.90
#   python regional_summary.py --threshold 70       # tau = 0.70
#   python regional_summary.py --model rf --threshold 95
#
# Output: printed table + <model>_regional_p<threshold>.csv (for the map later).

# --- Municipality (KOM) -> Region, per DST NUTS nomenclature ---------------
# Verify against the DST source before final use.
REGIONS = {
    "Hovedstaden": [
        101, 147, 151, 153, 155, 157, 159, 161, 163, 165, 167, 169, 173, 175,
        183, 185, 187, 190, 201, 210, 217, 219, 223, 230, 240, 250, 260, 270,
        400,  # Bornholm
    ],
    "Sjælland": [
        253, 259, 265, 269, 306, 316, 320, 326, 329, 330, 336, 340, 350, 360,
        370, 376, 390,
    ],
    "Syddanmark": [
        410, 420, 430, 440, 450, 461, 479, 480, 482, 492, 510, 530, 540, 550,
        561, 563, 573, 575, 580, 607, 621, 630,
    ],
    "Midtjylland": [
        615, 657, 661, 665, 671, 706, 707, 710, 727, 730, 740, 741, 746, 751,
        756, 760, 766, 779, 791,
    ],
    "Nordjylland": [
        773, 787, 810, 813, 820, 825, 840, 846, 849, 851, 860,
    ],
}
KOM_TO_REGION = {kom: reg for reg, koms in REGIONS.items() for kom in koms}

VALID_THRESHOLDS = (50, 70, 80, 90, 95)


def regional_summary(csv_path: str, threshold: int = 90) -> pd.DataFrame:
    """Sum municipal estimates into regions for the selected threshold (50/70/80/90/95)."""
    if threshold not in VALID_THRESHOLDS:
        raise ValueError(f"threshold must be one of {VALID_THRESHOLDS}")
    col = f"n_evs_p{threshold}"

    df = pd.read_csv(csv_path)
    df["region"] = df["KOM"].map(KOM_TO_REGION)

    unmapped = df.loc[df["region"].isna(), "KOM"].tolist()
    if unmapped:
        raise ValueError(f"KOM codes without region mapping: {unmapped}")

    out = (
        df.groupby("region")
          .agg(n_municipalities=("KOM", "size"),
               n_meters=("n_meters", "sum"),
               n_evs=(col, "sum"))
          .reindex(REGIONS.keys())
    )
    out["penetration_pct"] = (out["n_evs"] / out["n_meters"] * 100).round(2)
    # meter-weighted mean of the annual mean probability, for reference
    w = df["n_meters"] * df["mean_yearly_ev_probability"]
    out["weighted_mean_prob"] = (
        w.groupby(df["region"]).sum() / df.groupby("region")["n_meters"].sum()
    ).reindex(REGIONS.keys()).round(4)
    return out


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["rf", "xgb", "both"], default="both")
    ap.add_argument("--threshold", type=int, default=90, choices=VALID_THRESHOLDS)
    args = ap.parse_args()

    models = ["rf", "xgb"] if args.model == "both" else [args.model]
    for m in models:
        res = regional_summary(f"{m}_EV_area_distr.csv", args.threshold)
        print(f"\n=== {m.upper()} — regional estimates at P >= {args.threshold}% ===")
        print(res.to_string())
        out_csv = f"{m}_regional_p{args.threshold}.csv"
        res.to_csv(out_csv)
        print(f"saved {out_csv}")


usage: ipykernel_launcher.py [-h] [--model {rf,xgb,both}]
                             [--threshold {50,70,80,90,95}]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\chris\AppData\Roaming\jupyter\runtime\kernel-v373e723fbc9661b4a15a00b02a5d3c564f8fb4357.json


SystemExit: 2

C:\Users\chris\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


#### Regional Map

In [ ]:
# make_regional_map.py
# Single-hue choropleth ("heatmap") of estimated EV charger penetration for the
# five Danish regions, at a selectable probability threshold.
# Boundaries: official DAWA API (public). Uses regional_summary.py (same folder).
#
# Usage:
#   python make_regional_map.py                          # both models, tau=0.90
#   python make_regional_map.py --model xgb --threshold 70
#
# Requirements: pip install geopandas matplotlib requests



from regional_summary import regional_summary, VALID_THRESHOLDS

GEO_URL = "https://api.dataforsyningen.dk/regioner?format=geojson"

# DAWA region names -> names used in regional_summary.REGIONS
DAWA_TO_REGION = {
    "Nordjylland": "Nordjylland",
    "Midtjylland": "Midtjylland",
    "Syddanmark": "Syddanmark",
    "Hovedstaden": "Hovedstaden",
    "Sjælland": "Sjælland",
}


def load_boundaries() -> gpd.GeoDataFrame:
    r = requests.get(GEO_URL, timeout=120)
    r.raise_for_status()
    gdf = gpd.read_file(io.BytesIO(r.content))
    gdf["region"] = gdf["navn"].str.replace("Region ", "", regex=False).map(DAWA_TO_REGION)
    gdf["geometry"] = gdf.geometry.simplify(0.005)  # lighter rendering
    return gdf


def plot(models, threshold, cmap="Blues", out_pdf=None):
    gdf = load_boundaries()
    fig, axes = plt.subplots(1, len(models), figsize=(8 * len(models), 8))
    axes = [axes] if len(models) == 1 else list(axes)

    # shared colour scale across panels
    tables = {m: regional_summary(f"{m}_EV_area_distr.csv", threshold) for m in models}
    vmax = max(t["penetration_pct"].max() for t in tables.values())

    for ax, m in zip(axes, models):
        t = tables[m]
        g = gdf.merge(t, left_on="region", right_index=True)
        g.plot(column="penetration_pct", ax=ax, cmap=cmap, vmin=0, vmax=vmax,
               edgecolor="white", linewidth=0.8,
               legend=(m == models[-1]),
               legend_kwds={"label": f"EV charger penetration (% of meters), "
                                     f"$\\tau={threshold/100:.2f}$", "shrink": 0.6})
        # annotate each region with name and value
        for _, row in g.iterrows():
            c = row.geometry.representative_point()
            ax.annotate(f"{row['region']}\n{row['penetration_pct']:.1f}%",
                        xy=(c.x, c.y), ha="center", fontsize=9)
        ax.set_title({"rf": "Random Forest", "xgb": "XGBoost"}[m])
        ax.set_axis_off()

    plt.tight_layout()
    out_pdf = out_pdf or f"regional_map_p{threshold}.pdf"
    plt.savefig(out_pdf, bbox_inches="tight", dpi=300)
    print(f"saved {out_pdf}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", choices=["rf", "xgb", "both"], default="both")
    ap.add_argument("--threshold", type=int, default=90, choices=VALID_THRESHOLDS)
    ap.add_argument("--cmap", default="Blues")
    args = ap.parse_args()
    models = ["rf", "xgb"] if args.model == "both" else [args.model]
    plot(models, args.threshold, cmap=args.cmap)


#### Municipality Map

In [ ]:
# make_municipality_map.py
# Choropleth of estimated EV charger penetration per municipality (tau = 0.90).
# Requirements: pip install geopandas matplotlib requests
# Boundaries: official DAWA / Dataforsyningen API (public), joined on KOM code.



RF_CSV  = "rf_EV_area_distr.csv"
XGB_CSV = "xgb_EV_area_distr.csv"
OUT_PDF = "municipality_choropleth.pdf"
GEO_URL = "https://api.dataforsyningen.dk/kommuner?format=geojson"

# --- load model outputs -------------------------------------------------
def load(path, label):
    df = pd.read_csv(path)
    df["pen90"] = df["n_evs_p90"] / df["n_meters"] * 100.0
    return df[["KOM", "pen90"]].rename(columns={"pen90": label})

data = load(RF_CSV, "RF").merge(load(XGB_CSV, "XGB"), on="KOM")

# --- load municipal boundaries ------------------------------------------
# DAWA 'kode' is a zero-padded string ("0101"); convert to int to match KOM.
r = requests.get(GEO_URL, timeout=120)
r.raise_for_status()
gdf = gpd.read_file(io.BytesIO(r.content))
gdf["KOM"] = gdf["kode"].astype(int)
gdf = gdf.merge(data, on="KOM", how="left")
# lighter file / faster rendering (tolerance in degrees; boundaries stay visually intact)
gdf["geometry"] = gdf.geometry.simplify(0.002)

# --- plot ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 8))
vmax = gdf[["RF", "XGB"]].max().max()
for ax, col, title in zip(axes, ["RF", "XGB"], ["Random Forest", "XGBoost"]):
    gdf.plot(column=col, ax=ax, cmap="viridis", vmin=0, vmax=vmax,
             edgecolor="white", linewidth=0.3,
             legend=(col == "XGB"),
             legend_kwds={"label": "Estimated EV charger penetration (% of meters), $\\tau=0.90$",
                          "shrink": 0.6},
             missing_kwds={"color": "lightgrey"})
    ax.set_title(title)
    ax.set_axis_off()

plt.tight_layout()
plt.savefig(OUT_PDF, bbox_inches="tight", dpi=300)
print(f"saved {OUT_PDF}")
